# Layout

> The reference layout algorithm: sets in, grid cells out.

In [ ]:
#| default_exp layout

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *
import copy, random

This module is the **reference implementation** of ItemiSet's layout. The JavaScript library behind the website ([`js/itemiset.js`](https://github.com/adamcc/itemiset/blob/main/js/itemiset.js)) is a line-by-line port of it, and the [fixtures](fixtures.html) notebook checks that the two put every item in the same cell for a few hundred inputs. Everything that decides *which cell an item lands in* is therefore specified exactly here, including how ties are broken and how numbers are rounded.

## The template

One item is one cell, so a region's area is exactly its count. For up to three sets the cells are poured into a fixed template:

```
      +-- L --+--- M ---+-- R --+
      |  A    |  A∩B    |   B   |    upper zones (ragged top edge)
      |  A∩C  |  A∩B∩C  |  B∩C  |    lower zones (flat common baseline)
      +-------+---------+-------+
      |        C only           |    band under the baseline
      +-------------------------+
```

*A* is the left set, *B* the right set and *C* the bottom set. Each column group (L, M, R) is filled back and forth from the baseline up, lower zone first, so every zone is one connected piece. The cells of *C only* fill a band underneath. The layout search only has to choose which set goes at the bottom and three integer column widths, so it tries every candidate and keeps the best.

## Rules that keep Python and JavaScript identical

1. **Halves round up.** `round_half_up` is used wherever the layout rounds. Python's built-in `round` sends halves to the even neighbour, while JavaScript's `Math.round` sends them up.
2. **Scores are compared as integers**: the raw score times 10⁹, rounded half up. Two maths libraries can disagree in the last bit of `log`, and this keeps that from changing the winner.
3. **Candidates are tried in a fixed order**: bottom set in input order, total width from 1 upwards, then group widths in lexicographic order. The first candidate with the lowest score wins.
4. **Names sort naturally** (`natural_key`): ignoring case, with runs of digits compared as numbers, and ties kept in first-seen order.
5. **Lengths count characters (code points)**, not bytes.

In [ ]:
#| export
import math
import re
from dataclasses import dataclass, field
from itertools import combinations

In [ ]:
#| export
PLACEHOLDER = "∅"   # drawn in the single cell of an empty zone when `show_empty=True`
CELL_H_IN = 0.19    # cell height in inches, as drawn by the renderer

## Helpers

In [ ]:
#| export
def round_half_up(x: float) -> int:
    "Round `x` to the nearest integer, sending halves upwards (2.5 → 3, −2.5 → −2)."
    return math.floor(x + 0.5)

In [ ]:
test_eq([round_half_up(v) for v in (0.5, 1.5, 2.5, -0.5, -1.5, 2.49)], [1, 2, 3, 0, -1, 2])
test_eq([round(v) for v in (0.5, 1.5, 2.5)], [0, 2, 2])  # what we avoid

In [ ]:
#| export
_DIGITS = re.compile(r"([0-9]+)")

def natural_key(s: str) -> list:
    "Sort key that ignores case and compares runs of digits as numbers."
    return [int(t) if i % 2 else t.lower() for i, t in enumerate(_DIGITS.split(s))]

In [ ]:
test_eq(sorted(["CNOT10", "cnot2", "CNOT1", "AGO2", "ago1"], key=natural_key),
        ["ago1", "AGO2", "CNOT1", "cnot2", "CNOT10"])
test_eq(sorted(["b2", "B02", "b1"], key=natural_key), ["b1", "b2", "B02"])  # equal keys keep their order

Cells are as wide as the longest name needs. The layout uses the cell's width-to-height ratio so that the shape it aims for is the shape that gets drawn.

In [ ]:
#| export
def cell_size(names, fontsize: float = 6.5) -> tuple:
    "Width and height of one cell in inches, wide enough for the longest of `names`."
    maxlen = max([len(n) for n in names] + [4])
    return max(0.45, 0.0105 * fontsize * maxlen + 0.12), CELL_H_IN

def default_cell_aspect(names, fontsize: float = 6.5) -> float:
    "Width-to-height ratio of a cell that fits the longest of `names`."
    w, h = cell_size(names, fontsize)
    return w / h

In [ ]:
test_close(default_cell_aspect(["DDX6", "ANKHD1-EIF4EBP3"]), (0.0105 * 6.5 * 15 + 0.12) / 0.19)
test_close(default_cell_aspect(["a"]), 0.45 / 0.19)

## Cells and layouts

In [ ]:
#| export
@dataclass
class Cell:
    "One grid cell: where it is, which sets it belongs to, and which item it holds."
    x: float                   # left edge, in cell widths
    y: float                   # bottom edge, in cell heights; 0 is the baseline row, the band is below it
    zone: frozenset            # names of the sets this cell belongs to
    item: str | None = None    # None for filler and placeholder cells
    placeholder: bool = False  # True for the single ∅ cell of an empty zone

    @property
    def filler(self) -> bool:
        "Is this a blank cell added only to keep the bottom set in one piece?"
        return self.item is None and not self.placeholder

In [ ]:
#| export
@dataclass
class Layout:
    "A finished layout: its cells, plus how they were chosen."
    cells: list = field(default_factory=list)
    roles: dict = field(default_factory=dict)   # 'A' (left), 'B' (right), 'C' (bottom) -> set name
    widths: tuple = ()                          # widths of the left, middle and right column groups
    notes: list = field(default_factory=list)   # things the reader should be told
    names: list = field(default_factory=list)   # set names in input order, which is also colour order
    counts: dict = field(default_factory=dict)  # zone -> number of items
    plan: dict = field(default_factory=dict, repr=False)
    score: int = 0
    fig: object = field(default=None, repr=False)  # the figure, once drawn

    @property
    def n_items(self) -> int: return sum(1 for c in self.cells if c.item is not None)
    @property
    def n_filler(self) -> int: return sum(1 for c in self.cells if c.filler)

    def __repr__(self):
        roles = ", ".join(f"{k}={v!r}" for k, v in sorted(self.roles.items()))
        return f"Layout({self.n_items} items, {len(self.cells)} cells, roles: {roles}, widths={tuple(self.widths)})"

    def _repr_png_(self):
        if self.fig is None: return None
        import io
        buf = io.BytesIO()
        self.fig.savefig(buf, format="png", dpi=110)
        return buf.getvalue()

## Building blocks

In [ ]:
#| export
def _snake(w, n, inner_right):
    "(col, row) for `n` cells in a `w`-wide group, back and forth from the baseline, starting on the inner side."
    out = []
    r = 0
    while len(out) < n:
        cols = list(range(w))
        rtl = inner_right if r % 2 == 0 else not inner_right
        if rtl:
            cols.reverse()
        for c in cols:
            if len(out) == n:
                break
            out.append((c, r))
        r += 1
    return out

In [ ]:
test_eq(_snake(3, 7, inner_right=True), [(2, 0), (1, 0), (0, 0), (0, 1), (1, 1), (2, 1), (2, 2)])

In [ ]:
#| export
def _groups(roles):
    "Column groups as (name, lower zone, upper zone, is the inner side on the right?)."
    names = [roles[k] for k in sorted(roles)]
    if len(names) == 1:
        a, = names
        return [("L", frozenset([a]), None, True)]
    if len(names) == 2:
        a, b = names
        return [("L", frozenset([a]), None, True),
                ("M", frozenset([a, b]), None, False),
                ("R", frozenset([b]), None, False)]
    a, b, c = roles["A"], roles["B"], roles["C"]
    return [("L", frozenset([a, c]), frozenset([a]), True),
            ("M", frozenset([a, b, c]), frozenset([a, b]), False),
            ("R", frozenset([b, c]), frozenset([b]), False)]

def _compositions(W, k):
    "Every way to write `W` as an ordered sum of `k` non-negative integers, in lexicographic order."
    if k == 1:
        yield (W,)
        return
    for i in range(W + 1):
        for rest in _compositions(W - i, k - 1):
            yield (i,) + rest

In [ ]:
test_eq(list(_compositions(2, 3)), [(0,0,2), (0,1,1), (0,2,0), (1,0,1), (1,1,0), (2,0,0)])

`_plan` scores one candidate without building it. The score rewards a shape close to the target aspect ratio, and penalises empty space, filler cells and width:

$$\text{score} = 2\,\ln^2\!\frac{\text{aspect}}{\text{target}} + 3\cdot\text{empty fraction} + 1.5\cdot\text{filler cells} + 0.02\cdot\text{width}$$

In [ ]:
#| export
def _plan(counts, roles, widths, cell_aspect, target_aspect):
    "Evaluate one candidate. Returns (integer score, plan), or None if the widths don't fit the counts."
    groups = _groups(roles)
    ns = []
    for (g, lo, up, inner_right), w in zip(groups, widths):
        n = counts.get(lo, 0) + (counts.get(up, 0) if up else 0)
        if (n > 0) != (w > 0):
            return None
        ns.append(n)
    # a visible half-cell gap if L and R would otherwise touch with no M group between them
    gap = 0.5 if len(groups) == 3 and ns[0] and ns[2] and not ns[1] else 0.0
    xs, x = [], 0.0
    heights, c_intervals = [], []
    for i, ((g, lo, up, inner_right), w) in enumerate(zip(groups, widths)):
        if i == 2:
            x += gap
        xs.append(x)
        heights.append(math.ceil(ns[i] / w) if w else 0)
        n_lo = counts.get(lo, 0)
        if "C" in roles and w and n_lo:
            k = min(n_lo, w)
            c_intervals.append((x + w - k, x + w) if inner_right else (x, x + k))
        x += w
    total_w = sum(widths) + gap
    hg = max(heights) if heights else 0

    band = None
    filler = 0
    if "C" in roles:
        n_c = counts.get(frozenset([roles["C"]]), 0)
        iv = sorted(c_intervals)
        merged = []
        for a, b in iv:
            if merged and a <= merged[-1][1] + 1e-9:
                merged[-1] = (merged[-1][0], max(merged[-1][1], b))
            else:
                merged.append((a, b))
        connected = len(merged) <= 1
        if merged:
            s0, s1 = merged[0][0], merged[-1][1]
        else:
            s0 = s1 = total_w / 2
        span = s1 - s0
        need = 0 if connected else math.ceil(span - 1e-9)
        n_band = max(n_c, need)
        if n_band:
            filler = n_band - n_c
            rows_target = max(1, round_half_up(hg * 0.25))
            bw = max(math.ceil(span - 1e-9), math.ceil(n_band / rows_target), 1)
            bw = min(bw, max(1, math.floor(total_w + 1e-9)), n_band) if total_w >= 1 else bw
            bw = max(bw, min(n_band, math.ceil(span - 1e-9)))
            centre = (s0 + s1) / 2
            x0 = min(max(centre - bw / 2, 0.0), max(0.0, total_w - bw))
            if merged:                       # snap so band cells line up with the columns above
                x0 = round_half_up(x0 - merged[0][0]) + merged[0][0]
                x0 = min(max(x0, 0.0), max(0.0, total_w - bw))
            rows = math.ceil(n_band / bw)
            vgap = 0 if merged else 0.4      # C shares nothing with A or B: float it below
            band = dict(x0=x0, bw=bw, rows=rows, n=n_band, vgap=vgap)

    band_rows = (band["rows"] + band["vgap"]) if band else 0
    H = hg + band_rows
    ncells = sum(counts.values()) + filler
    aspect = (total_w * cell_aspect) / max(H, 1e-9)
    empty = max(0.0, total_w * H - ncells) / max(total_w * H, 1)
    l = math.log(aspect / target_aspect)
    raw = 2.0 * (l * l) + 3.0 * empty + 1.5 * filler + 0.02 * total_w
    return round_half_up(raw * 1e9), dict(widths=widths, xs=xs, heights=heights, band=band,
                                           filler=filler, gap=gap, total_w=total_w, H=H, hg=hg)

## Laying out sets

In [ ]:
#| export
def compute_layout(sets: dict, show_empty=False, bottom=None, cell_aspect=None,
                   target_aspect=2.2, max_width=None) -> Layout:
    "Lay out 1–3 sets given as `{name: [items]}`. This is the core algorithm; most callers want `layout_sets`."
    names = list(sets)
    membership = {}
    for s in names:
        for item in sets[s]:
            membership.setdefault(item, set()).add(s)
    zones = {}
    for item, ss in membership.items():
        zones.setdefault(frozenset(ss), []).append(item)
    counts = {z: len(v) for z, v in zones.items()}
    if cell_aspect is None:
        cell_aspect = default_cell_aspect(list(membership))

    notes = []
    placeholders = set()
    if show_empty:
        for k in range(1, len(names) + 1):
            for combo in combinations(names, k):
                z = frozenset(combo)
                if not counts.get(z):
                    counts[z] = 1
                    placeholders.add(z)
        if placeholders:
            notes.append(f"{len(placeholders)} empty zone(s) shown as '{PLACEHOLDER}' "
                         "placeholder cells (1 cell each; not area-proportional).")
    else:
        for s in names:
            if not any(s in z for z in counts):
                notes.append(f"Set '{s}' is empty and is not drawn.")

    # candidate role assignments: each set in turn at the bottom, in input order
    if len(names) == 3:
        cands = []
        for c in names:
            if bottom and c != bottom:
                continue
            a, b = [n for n in names if n != c]
            cands.append({"A": a, "B": b, "C": c})
        if not cands:
            raise ValueError(f"bottom set {bottom!r} is not one of {names}")
    elif len(names) == 2:
        cands = [{"A": names[0], "B": names[1]}]
    else:
        cands = [{"A": names[0]}]

    n_total = sum(counts.values())
    wmax = max_width or max(4, min(60, int(math.sqrt(n_total * target_aspect / cell_aspect) * 3) + 2))
    best = None
    for roles in cands:
        k = len(_groups(roles))
        for W in range(1, wmax + 1):
            for widths in _compositions(W, k):
                r = _plan(counts, roles, widths, cell_aspect, target_aspect)
                if r and (best is None or r[0] < best[0]):
                    best = (r[0], roles, r[1])
    if best is None:
        raise RuntimeError("No valid layout found (are all sets empty?)")
    score, roles, plan = best

    # build the cells
    lay = Layout(roles=roles, widths=plan["widths"], notes=notes, names=names, plan=plan, score=score)
    zone_cells = {}
    for (g, lo, up, inner_right), w, x0 in zip(_groups(roles), plan["widths"], plan["xs"]):
        n_lo = counts.get(lo, 0)
        n_up = counts.get(up, 0) if up else 0
        for i, (c, r) in enumerate(_snake(w, n_lo + n_up, inner_right)):
            z = lo if i < n_lo else up
            zone_cells.setdefault(z, []).append(Cell(x0 + c, r, z))
    band = plan["band"]
    if band:
        zc = frozenset([roles["C"]])
        bw, n = band["bw"], band["n"]
        full_rows = n // bw
        rem = n - full_rows * bw
        for i in range(full_rows):
            for c in range(bw):
                zone_cells.setdefault(zc, []).append(Cell(band["x0"] + c, -1 - i - band["vgap"], zc))
        if rem:
            off = (bw - rem) // 2
            for c in range(rem):
                zone_cells.setdefault(zc, []).append(
                    Cell(band["x0"] + off + c, -1 - full_rows - band["vgap"], zc))
        if plan["filler"]:
            notes.append(f"{plan['filler']} blank filler cell(s) added to '{roles['C']} only' "
                         "to keep that set connected.")

    # items go in reading order (down each column, left to right), sorted naturally
    for z, cells in zone_cells.items():
        cells.sort(key=lambda c: (c.x, -c.y))
        if z in placeholders:
            cells[0].placeholder = True
        else:
            items = sorted(zones.get(z, []), key=natural_key)
            for cell, item in zip(cells, items):
                cell.item = item
        lay.cells.extend(cells)
    lay.counts = {z: len(v) for z, v in zones.items()}
    return lay

In [ ]:
#| export
def layout_sets(sets: dict, show_empty=False, bottom=None, cell_aspect=None, target_aspect=2.2,
                fontsize=6.5, max_width=None) -> Layout:
    "Lay out 1–3 named sets. Items are converted to text, duplicates dropped, and empty sets left out with a note."
    sets = {str(k): list(dict.fromkeys(map(str, v))) for k, v in sets.items()}
    if not 1 <= len(sets) <= 3:
        raise ValueError(f"ItemiSet draws 1 to 3 sets; got {len(sets)}: {list(sets)}")
    dropped = [k for k, v in sets.items() if not v]
    active = {k: v for k, v in sets.items() if v}
    if not active:
        raise ValueError("All sets are empty, so there is nothing to draw.")
    if bottom in dropped:
        bottom = None
    if cell_aspect is None:
        cell_aspect = default_cell_aspect([i for v in active.values() for i in v], fontsize)
    lay = compute_layout(active, show_empty=show_empty, bottom=bottom, cell_aspect=cell_aspect,
                         target_aspect=target_aspect, max_width=max_width)
    lay.notes[:0] = [f"Set '{k}' has no members and was left out of the diagram." for k in dropped]
    lay.names = list(sets)
    return lay

A small example: three sets with every zone filled.

In [ ]:
demo = {"Left":   ["a1", "a2", "a3", "ab1", "ab2", "ac1", "abc1", "abc2"],
        "Right":  ["b1", "b2", "ab1", "ab2", "bc1", "abc1", "abc2"],
        "Bottom": ["c1", "c2", "c3", "ac1", "bc1", "abc1", "abc2"]}
lay = layout_sets(demo)
lay

Layout(14 items, 14 cells, roles: A='Right', B='Bottom', C='Left', widths=(1, 1, 1))

In [ ]:
#| export
def text_grid(lay: Layout, width: int = 6) -> str:
    "A plain-text picture of a layout, one name per cell (`·` for filler)."
    xs = sorted({c.x for c in lay.cells}); ys = sorted({c.y for c in lay.cells}, reverse=True)
    pos = {(c.x, c.y): c for c in lay.cells}
    lines = []
    for y in ys:
        row = []
        for x in xs:
            c = pos.get((x, y))
            t = "" if c is None else (PLACEHOLDER if c.placeholder else ("·" if c.filler else c.item))
            row.append(t[:width].ljust(width))
        lines.append(" ".join(row).rstrip())
    return "\n".join(lines)

In [ ]:
print(text_grid(lay))

b1            c1
b2     bc1    c2
ab1    abc1   c3
ab2    abc2   ac1
a1     a2     a3


In [ ]:
test_eq(lay.n_items, 14)
test_eq(lay.roles, {"A": "Right", "B": "Bottom", "C": "Left"})
test_eq(len(lay.cells), 14)

Empty sets are dropped (and reported), and `show_empty=True` gives every empty zone one ∅ cell, which turns the Euler diagram into a Venn diagram:

In [ ]:
lay2 = layout_sets({"A": ["x", "y", "z"], "B": [], "C": ["z", "w"]})
test_eq(lay2.notes, ["Set 'B' has no members and was left out of the diagram."])
lay3 = layout_sets({"A": ["x", "y"], "B": ["y", "z"], "C": ["q"]}, show_empty=True)
print(text_grid(lay3)); print(lay3.notes)
test_eq(sum(c.placeholder for c in lay3.cells), 3)  # A∩C, B∩C and A∩B∩C are empty

z      ∅      q
y      ∅      ∅
       x
["3 empty zone(s) shown as '∅' placeholder cells (1 cell each; not area-proportional)."]


## Records

`layout_record` turns a layout into plain JSON-ready data. The JavaScript tests compare their output with these records.

In [ ]:
#| export
def layout_record(lay: Layout) -> dict:
    "Plain, JSON-ready summary of `lay`: roles, widths, notes, score and every cell as [x, y, zone, item, placeholder]."
    cells = sorted(lay.cells, key=lambda c: (c.x, -c.y))
    return dict(roles=dict(sorted(lay.roles.items())), widths=list(lay.widths), notes=list(lay.notes),
                score=lay.score,
                cells=[[c.x, c.y, [n for n in lay.names if n in c.zone], c.item, c.placeholder] for c in cells])

In [ ]:
rec = layout_record(lay)
rec["cells"][:3]

[[0.0, 3, ['Right'], 'b1', False],
 [0.0, 2, ['Right'], 'b2', False],
 [0.0, 1, ['Left', 'Right'], 'ab1', False]]

In [ ]:
#| export
def zone_rows(lay: Layout, categories: dict | None = None) -> list:
    "One row per item: the item, its zone, a 0/1 column per set, and its category if any."
    categories = categories or {}
    rows = [{"item": c.item, "zone": " & ".join(s for s in lay.names if s in c.zone),
             **{s: int(s in c.zone) for s in lay.names}, "category": categories.get(c.item, "")}
            for c in lay.cells if c.item is not None]
    return sorted(rows, key=lambda r: (r["zone"], natural_key(r["item"])))

In [ ]:
zone_rows(lay)[:3]

[{'item': 'c1',
  'zone': 'Bottom',
  'Left': 0,
  'Right': 0,
  'Bottom': 1,
  'category': ''},
 {'item': 'c2',
  'zone': 'Bottom',
  'Left': 0,
  'Right': 0,
  'Bottom': 1,
  'category': ''},
 {'item': 'c3',
  'zone': 'Bottom',
  'Left': 0,
  'Right': 0,
  'Bottom': 1,
  'category': ''}]

## Checking a layout

`check_layout` tests the five properties every layout must have. The checks run on a fine raster (each cell is 4 × 5 squares) because the layout can shift a group by half a cell and float a detached band 0.4 of a row below the rest.

In [ ]:
#| export
_SX, _SY = 4, 5   # raster squares per cell, across and up

def _squares(c):
    x0, y0 = round_half_up(c.x * _SX), round_half_up(c.y * _SY)
    return [(x, y) for x in range(x0, x0 + _SX) for y in range(y0, y0 + _SY)]

def _connected(sq: set) -> bool:
    if not sq: return True
    start = next(iter(sq)); seen = {start}; stack = [start]
    while stack:
        x, y = stack.pop()
        for n in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
            if n in sq and n not in seen:
                seen.add(n); stack.append(n)
    return len(seen) == len(sq)

def _enclosed(sq: set) -> int:
    "Number of squares not in `sq` that can't reach the outside."
    if not sq: return 0
    xs = [x for x, _ in sq]; ys = [y for _, y in sq]
    x0, x1, y0, y1 = min(xs) - 1, max(xs) + 1, min(ys) - 1, max(ys) + 1
    seen = {(x0, y0)}; stack = [(x0, y0)]
    while stack:
        x, y = stack.pop()
        for n in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
            if x0 <= n[0] <= x1 and y0 <= n[1] <= y1 and n not in sq and n not in seen:
                seen.add(n); stack.append(n)
    return (x1 - x0 + 1) * (y1 - y0 + 1) - len(sq) - len(seen)

In [ ]:
#| export
CHECKS = ("One cell per item", "Each set is one piece", "No set has a hole",
          "Each zone is one piece", "No stray white pockets")

def check_layout(lay: Layout, sets: dict | None = None) -> dict:
    "Test the five properties every layout must have. Returns {check name: passed?}."
    allsq, overlap = set(), False
    for c in lay.cells:
        for s in _squares(c):
            overlap |= s in allsq
            allsq.add(s)
    items = [c.item for c in lay.cells if c.item is not None]
    one = not overlap and len(items) == len(set(items))
    if sets is not None:
        one &= set(items) == {str(i) for v in sets.values() for i in v}
    set_sq = {s: {q for c in lay.cells if s in c.zone for q in _squares(c)} for s in lay.names}
    zone_sq = {}
    for c in lay.cells:
        zone_sq.setdefault(c.zone, set()).update(_squares(c))
    return dict(zip(CHECKS, [
        one,
        all(_connected(q) for q in set_sq.values()),
        all(_enclosed(q) == 0 for q in set_sq.values()),
        all(_connected(q) for q in zone_sq.values()),
        _enclosed(allsq) == 0,
    ]))

def assert_valid(lay: Layout, sets: dict | None = None):
    "Raise `AssertionError` naming any check that `lay` fails."
    bad = [k for k, ok in check_layout(lay, sets).items() if not ok]
    if bad: raise AssertionError(f"Layout fails: {', '.join(bad)}")

In [ ]:
check_layout(lay, demo)

{'One cell per item': True,
 'Each set is one piece': True,
 'No set has a hole': True,
 'Each zone is one piece': True,
 'No stray white pockets': True}

The checks do catch broken layouts. Here we punch a hole in one set by moving a cell, and split another:

In [ ]:
broken = copy.deepcopy(lay)
broken.cells[0].x += 20
test_eq(check_layout(broken)["Each zone is one piece"] and check_layout(broken)["Each set is one piece"], False)

### Random layouts

Every layout of a few hundred random inputs must pass all five checks. The inputs cover 1–3 sets, empty zones, empty sets, Venn mode, a forced bottom set and a range of shapes.

In [ ]:
def random_sets(rng, k=None, maxn=15):
    k = k or rng.randint(1, 3)
    names = ["A", "B", "C"][:k]
    sets = {n: [] for n in names}
    for m in range(1, 2**k):
        n = 0 if rng.random() < 0.35 else rng.randint(1, maxn)
        members = [names[i] for i in range(k) if m >> i & 1]
        for j in range(n):
            for s in members: sets[s].append(f"{''.join(members).lower()}{j}")
    return sets

rng = random.Random(1)
n_ok = n_filler = 0
for t in range(250):
    sets = random_sets(rng)
    if not any(sets.values()): continue
    opts = dict(show_empty=rng.random() < 0.3, target_aspect=rng.choice([0.6, 1.2, 2.2, 3.0]))
    if len(sets) == 3 and rng.random() < 0.3: opts["bottom"] = rng.choice(list(sets))
    lay_r = layout_sets(sets, **opts)
    assert_valid(lay_r, {k: v for k, v in sets.items()})
    n_ok += 1; n_filler += lay_r.n_filler > 0
print(f"{n_ok} random layouts passed every check; {n_filler} needed filler cells.")

216 random layouts passed every check; 4 needed filler cells.


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()